# Pentora CART — Colab runner

Autonomous **Continuous Automated Red Teaming** on a local LLM. Nothing leaves this VM.

**Default model:** `hf.co/HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive:Q4_K_M` — an uncensored community fine-tune of Qwen3.5-9B (Apache-2.0, GGUF). The LLM here only classifies a captured transaction into a vulnerability category to route playbooks — it never mints a finding, a deterministic validator does (see `docs/cart-engine.md`). Safety-tuned base models sometimes refuse or hedge on "does this look exploitable" framing even for benign, authorized security testing; this avoids that. Swap the `MODEL` line in cells 2a/2b/4 for any other Ollama tag (e.g. `qwen3.5:9b`) if you'd rather use the stock model.

Run the cells top to bottom:
1. Install the engine (from the `feature/engine-core` branch)
2. Install + start Ollama, pull the model (its own cell — shows live download progress), then sanity-check it answers
3. **Self-contained demo** — the engine proves a real IDOR against a vulnerable API spun up inside this VM (no external target, fully legal)
4. Template to scan **your own authorized target** from a HAR export

> Only test systems you own or are authorized to test.

## 1. Install the engine

In [ ]:
# 1) Install the CART engine from the branch (NOT PyPI — the engine isn't released yet).
#    Extras MUST be on the SAME git URL: a separate `pip install "pentora[engine,capture]"`
#    line with no git URL resolves against PyPI instead of this branch and can silently miss
#    a brand-new dependency. --force-reinstall so re-running this cell always picks up the
#    latest commit instead of a cached install.
!pip -q install --force-reinstall "git+https://github.com/sohan-a11y/pentora.git@feature/engine-core#egg=pentora[engine,capture]"
print("[1/2] engine installed")

# 1b) FAST self-test: confirm async-primitive calls work from inside Colab's running event
#     loop BEFORE spending minutes on the Ollama/model setup below.
import asyncio
from pentora.engine.asyncrun import run_sync

async def _probe():
    await asyncio.sleep(0)
    return "ok"

async def _drive_probe():
    asyncio.get_running_loop()
    return run_sync(_probe())

assert asyncio.run(_drive_probe()) == "ok"
print("[2/2] async-in-notebook self-test passed")


## 2. Install + start Ollama (optional)

In [ ]:
# 2) Install Ollama and start its server (OPTIONAL — the deterministic playbooks
#    in cell 3 work without it; the LLM only adds smarter 'where to look' routing).
import os, time, shutil, subprocess, urllib.request
# the current Ollama installer unpacks a zstd archive — Colab lacks zstd, so install it first
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
!curl -fsSL https://ollama.com/install.sh | sh
ollama = shutil.which("ollama")
assert ollama, "ollama still not on PATH — check the install output above (usually a zstd issue)"
print("ollama binary:", ollama)

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
# start_new_session=True detaches it from this cell's process group, so interrupting a LATER
# cell (e.g. a slow pull) won't kill the server.
subprocess.Popen([ollama, "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT,
                  start_new_session=True)
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        print("ollama server is up"); break
    except Exception:
        time.sleep(2)
else:
    print("server did NOT come up — tail of /tmp/ollama.log:")
    print(open("/tmp/ollama.log").read()[-1500:])
    raise RuntimeError("ollama server failed to start — see log above")


## 2a. Pull the model (own cell — shows live progress)

In [ ]:
# 2a) Pull the model — run as ITS OWN cell with shell magic (`!`), not subprocess.run().
#     subprocess.run() has no real terminal attached, so Ollama's progress bar gets buffered
#     and NOTHING prints for minutes — it looks frozen even though it's downloading fine.
#     `!` gives it a real terminal, so you see live "pulling manifest... 42%..." progress.
!ollama pull hf.co/HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive:Q4_K_M


## 2b. Sanity-check the model

In [ ]:
# 2b) Sanity-check the model (the CORRECT Ollama call — messages is a LIST of dicts,
#     not a bare string, which is what caused the earlier ValueError). num_ctx is capped at 4096:
#     this model natively supports up to 262K context, and letting the KV cache auto-size to
#     that would eat far more VRAM than the engine's short JSON classification prompts need.
import urllib.request, json
payload = {"model": "hf.co/HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive:Q4_K_M", "messages": [{"role": "user", "content": "reply with just: ok"}],
           "stream": False, "options": {"temperature": 0, "num_ctx": 4096}}
req = urllib.request.Request("http://127.0.0.1:11434/api/chat",
                            data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
print("model says:", json.loads(urllib.request.urlopen(req, timeout=300).read())["message"]["content"])
!ollama ps    # should show 100% GPU on your T4


## 3. Self-contained demo — prove a finding with zero external dependencies

In [ ]:
# 3) SELF-CONTAINED DEMO — no external target needed, fully legal.
# Spins a deliberately-vulnerable JWT/IDOR API inside this VM and lets the engine PROVE an
# IDOR: as user_b, request user_a's order and confirm user_a's private data comes back.
import base64, hashlib, hmac, json, threading
from http.server import BaseHTTPRequestHandler, HTTPServer

from pentora.engine import (
    Blackboard, DeterministicValidator, Governor, Hypothesis, RunScope,
)
from pentora.engine.playbook_idor import IdorPlaybookContext, run_idor_playbook

SECRET = "s3cr3t"
ORDERS = {"1": ("user_a", "ALPHA-secret-order-one"), "2": ("user_b", "BETA-secret-order-two")}

def b64u(b): return base64.urlsafe_b64encode(b).rstrip(b"=").decode()

def make_jwt(sub):
    h = b64u(json.dumps({"alg": "HS256", "typ": "JWT"}).encode())
    p = b64u(json.dumps({"sub": sub, "role": "user"}, separators=(",", ":")).encode())
    sig = b64u(hmac.new(SECRET.encode(), f"{h}.{p}".encode(), hashlib.sha256).digest())
    return f"{h}.{p}.{sig}"

def token_sub(token):
    try:
        h, p, sig = token.split(".")
        exp = b64u(hmac.new(SECRET.encode(), f"{h}.{p}".encode(), hashlib.sha256).digest())
        if not hmac.compare_digest(exp, sig): return None
        return str(json.loads(base64.urlsafe_b64decode(p + "=" * (-len(p) % 4)))["sub"])
    except Exception:
        return None

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        oid = self.path.rstrip("/").rsplit("/", 1)[-1]
        auth = self.headers.get("Authorization", "")
        sub = token_sub(auth[7:].strip() if auth[:7].lower() == "bearer " else "")
        if sub is None:
            code, body = 401, "unauthorized"
        elif oid not in ORDERS:
            code, body = 404, "not found"
        else:
            # VULNERABLE: returns any order by id, ignoring ownership
            code, body = 200, ORDERS[oid][1]
        data = body.encode()
        self.send_response(code); self.send_header("Content-Length", str(len(data))); self.end_headers()
        self.wfile.write(data)
    def log_message(self, *a): pass

srv = HTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=srv.serve_forever, daemon=True).start()
base = f"http://127.0.0.1:{srv.server_address[1]}"
print("vulnerable demo API listening at", base)

bb = Blackboard()
hyp = bb.assert_fact(Hypothesis(source="demo", claim="idor"))
status = run_idor_playbook(IdorPlaybookContext(
    bb=bb, governor=Governor(), validator=DeterministicValidator(), hypothesis=hyp,
    victim_url=f"{base}/api/orders/1", victim_token=make_jwt("user_a"),
    attacker_token=make_jwt("user_b"), attacker_own_url=f"{base}/api/orders/2",
    scope=RunScope(include=["127.0.0.1"], read_only=True),
))
print("playbook status:", status)
for f in bb.query("finding"):
    print(f"\nFINDING: {f.title}  CVSS {f.cvss_score} ({f.severity})")
    print("  evidence:", f.evidence)
    print("  PoC:", f.poc)
if not bb.query("finding"):
    print("no finding (unexpected for this vulnerable demo)")


## 4. Scan your own authorized target (HAR upload)

In [ ]:
# 4) SCAN YOUR OWN AUTHORIZED TARGET
# Capture traffic in your browser (DevTools -> Network -> Save all as HAR), upload it here,
# then run the full autonomous engine.  ONLY scan systems you own or are authorized to test.
from google.colab import files
from pentora.engine.app import start

TARGET = "https://your-app.example.com"     # <-- edit me (authorized target only)

up = files.upload()                          # pick your traffic.har
har_path = next(iter(up))

e = start(target=TARGET, model="hf.co/HauhauCS/Qwen3.5-9B-Uncensored-HauhauCS-Aggressive:Q4_K_M", scope_hosts=[TARGET.split("//")[-1].split("/")[0]])
e.ingest_har(har_path)
summary = e.run()
print(summary)
print(e.report_markdown("pentora-cart-report.md"))

# Continuous mode: save today's findings as the baseline, then later diff to see only NEW ones.
# e.save_baseline("baseline.json")
